# Equation discovery: which terms govern the data?

**Book:** §4.7, Figure 4.3(d) &nbsp;·&nbsp; `ch04/equation_discovery.ipynb`

We are handed noisy measurements $u(x,t)$ and asked not to solve a PDE but to *find* it. Posit a
**library** of candidate terms and let the data choose:

$$u_t \;=\; \xi_1 u + \xi_2 u_x + \xi_3 u_{xx} + \xi_4\, u u_x + \xi_5 u^2 .$$

$$\mathcal L=\overline{\Big(u_t-\textstyle\sum_j \xi_j\,\theta_j\Big)^2}
\;+\;10\,\overline{(u-u^{\rm obs})^2}\;+\;\underbrace{10^{-4}\!\sum_j|\xi_j|}_{\text{L1: prefer few terms}}$$

The $\xi_j$ are **trainable scalars alongside the network weights**. The L1 penalty is what
delivers a *parsimonious* law rather than a five-term fit that happens to work.

Truth (hidden from the solver): $c=1$, $\nu=0.02$, i.e. $\xi=(0,-1,0.02,0,0)$.

In [ ]:
import time
import numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt
np.random.seed(0); torch.manual_seed(0)
def g1(f, x): return torch.autograd.grad(f, x, torch.ones_like(f), create_graph=True)[0]
def mlp(s):
    L = []
    for i in range(len(s)-1):
        L.append(nn.Linear(s[i], s[i+1]))
        if i < len(s)-2: L.append(nn.Tanh())
    return nn.Sequential(*L)

C, NU, L, T, X0, S0 = 1.0, 0.02, 2.0, 1.0, 0.4, 0.12       # the TRUTH, hidden from the solver
uex = lambda x, t: (S0/np.sqrt(S0**2+2*NU*t))*np.exp(-((x-X0-C*t)**2)/(2*(S0**2+2*NU*t)))

ND = 3000
xd, td = np.random.rand(ND)*L, np.random.rand(ND)*T
ud = uex(xd, td) + 0.005*np.random.randn(ND)               # noisy scattered measurements
xd_t = torch.tensor(xd, dtype=torch.float32).reshape(-1,1)
td_t = torch.tensor(td, dtype=torch.float32).reshape(-1,1)
ud_t = torch.tensor(ud, dtype=torch.float32).reshape(-1,1)

net = mlp([2,64,64,64,1])
xi  = nn.Parameter(torch.zeros(5,1))                       # the five unknown coefficients
opt = torch.optim.Adam(list(net.parameters()) + [xi], 2e-3)

t0 = time.perf_counter()
for e in range(12000):
    if e == 8000:
        for g in opt.param_groups: g['lr'] = 5e-4
    opt.zero_grad()
    x = (torch.rand(2000,1)*L).requires_grad_(True)
    t = (torch.rand(2000,1)*T).requires_grad_(True)
    u = net(torch.cat([x,t],1))
    ux = g1(u,x); uxx = g1(ux,x); ut = g1(u,t)
    lib = torch.cat([u, ux, uxx, u*ux, u**2], 1)           # the candidate library
    loss = ((ut - lib@xi)**2).mean() \
         + 10*((net(torch.cat([xd_t,td_t],1)) - ud_t)**2).mean() \
         + 1e-4*xi.abs().sum()                             # L1 -> parsimony
    loss.backward(); opt.step()
print(f'training: {time.perf_counter()-t0:.0f} s')

xiv  = xi.detach().numpy().ravel()
true = np.array([0, -C, NU, 0, 0])
names = ['u', 'u_x', 'u_xx', 'u u_x', 'u^2']
print(f'\n{"term":>7}  {"discovered":>11}  {"true":>7}')
for n_, d_, t_ in zip(names, xiv, true):
    print(f'{n_:>7}  {d_:11.4f}  {t_:7.4f}')
print(f'\nDiscovered law:  u_t = {xiv[1]:+.4f} u_x {xiv[2]:+.4f} u_xx')
print(f'True law      :  u_t = {-C:+.4f} u_x {NU:+.4f} u_xx')

tex = ['$u$', '$u_x$', '$u_{xx}$', '$u\\,u_x$', '$u^2$']
w = 0.38; idx = np.arange(5)
plt.figure(figsize=(8,4.3))
plt.bar(idx-w/2, true, w, label='true',       color='tab:green', alpha=.7)
plt.bar(idx+w/2, xiv,  w, label='discovered', color='tab:red',   alpha=.85)
plt.xticks(idx, tex); plt.axhline(0, color='k', lw=.7)
plt.ylabel('coefficient'); plt.legend(fontsize=9, loc='lower right'); plt.grid(alpha=.3, axis='y')
plt.title('Equation discovery: the L1 penalty silences the spurious terms')
for i, d_ in enumerate(xiv):
    va, off = ('top', -0.04) if d_ < -0.3 else ('bottom', 0.03)
    plt.text(i+w/2, d_+off, f'{d_:.3f}', ha='center', va=va, fontsize=8)
plt.tight_layout(); plt.show()